In [5]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

# Project paths
BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR / "my-own-experiement"

DATA_DIR = BASE_DIR / "data"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Config
MODEL_NAME = "buffalo_l"        
DET_THRESH = 0.5
QUALITY_DET_SCORE = 0.60           # keep better face detections
COSINE_THRESHOLD = 0.60            # will tune later
ENROLL_PER_PERSON = 30             # images used to build each person's template

# Customize person roles here
ROLE_BY_PERSON = {
    "holland": "student",
    "roth": "student",
    "vit": "student",
    "sopheak": "teacher",
    "vannet": "teacher",
}

def get_person_role(person_name, role_map=ROLE_BY_PERSON, default_role="unknown"):
    return role_map.get(str(person_name).strip().lower(), default_role)

print(f"DATA_DIR: {DATA_DIR}")
print(f"ARTIFACTS_DIR: {ARTIFACTS_DIR}")
print("Configured roles:", ROLE_BY_PERSON)

DATA_DIR: c:\Users\chans\Documents\Sopheak\PersonalProject\fullstack\face-regcon-attendance\face_recognition_attendance_system\my-own-experiement\data
ARTIFACTS_DIR: c:\Users\chans\Documents\Sopheak\PersonalProject\fullstack\face-regcon-attendance\face_recognition_attendance_system\my-own-experiement\artifacts
Configured roles: {'holland': 'student', 'roth': 'student', 'vit': 'student', 'sopheak': 'teacher', 'vannet': 'teacher'}


In [6]:
from insightface.app import FaceAnalysis

def l2_normalize(vec):
    vec = np.asarray(vec, dtype=np.float32)
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm

model = FaceAnalysis(name=MODEL_NAME, providers=["CPUExecutionProvider"])
model.prepare(ctx_id=-1, det_size=(640, 640), det_thresh=DET_THRESH)


def extract_best_embedding(image_path):
    """Return (normalized_embedding, det_score) for best detected face, or None."""
    img = cv2.imread(str(image_path))
    if img is None:
        return None

    faces = model.get(img, max_num=0)
    if not faces:
        return None

    best_face = max(faces, key=lambda f: float(f["det_score"]))
    emb = l2_normalize(best_face["embedding"])
    det_score = float(best_face["det_score"])
    return emb, det_score

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\chans/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\chans/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\chans/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\chans/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\chans/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.

In [7]:
# check dataset folders
rows = []
for person_dir in sorted(DATA_DIR.iterdir()):
    if not person_dir.is_dir():
        continue
    person_name = person_dir.name
    person_role = get_person_role(person_name)
    for img_path in sorted(person_dir.glob("*.jpg")):
        rows.append({
            "person": person_name,
            "role": person_role,
            "image_path": img_path,
        })

dataset_df = pd.DataFrame(rows)
print(f"Total images found: {len(dataset_df)}")
print(dataset_df.groupby(["person", "role"]).size())
dataset_df.head()

Total images found: 367
person   role   
holland  student    49
roth     student    91
sopheak  teacher    78
vannet   teacher    72
vit      student    77
dtype: int64


,person,role,image_path
0,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...
1,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...
2,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...
3,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...
4,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...


In [8]:
# Extract one embedding per image
records = []
for row in dataset_df.itertuples(index=False):
    out = extract_best_embedding(row.image_path)
    if out is None:
        continue

    emb, det_score = out
    records.append({
        "person": row.person,
        "role": row.role,
        "image_path": str(row.image_path),
        "det_score": det_score,
        "embedding": emb,
    })

emb_df = pd.DataFrame(records)
print(f"Embeddings extracted: {len(emb_df)} / {len(dataset_df)}")
print(emb_df.groupby(["person", "role"]).size())
emb_df.head()

c:\Users\chans\Documents\Sopheak\PersonalProject\fullstack\face-regcon-attendance\face_recognition_attendance_system\my-own-experiement\env\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


Embeddings extracted: 367 / 367
person   role   
holland  student    49
roth     student    91
sopheak  teacher    78
vannet   teacher    72
vit      student    77
dtype: int64


,person,role,image_path,det_score,embedding
0,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...,0.824249,"[0.045902308, -0.044096492, -0.026930269, -0.0..."
1,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...,0.812640,"[-2.4902847e-05, 0.014311185, -0.07868694, -0...."
2,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...,0.891258,"[0.052478112, 0.001045927, -0.04254915, -0.001..."
3,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...,0.833919,"[-0.014520578, -0.021186246, -0.018602071, -0...."
4,holland,student,c:\Users\chans\Documents\Sopheak\PersonalProje...,0.777937,"[0.042174462, -0.030343218, -0.07517657, -0.03..."


In [9]:
# Keep good-quality detections, then split enrollment/evaluation
quality_df = emb_df[emb_df["det_score"] >= QUALITY_DET_SCORE].copy()
if quality_df.empty:
    quality_df = emb_df.copy()

enroll_parts = []
eval_parts = []

for person, group in quality_df.sort_values("image_path").groupby("person"):
    n_enroll = min(ENROLL_PER_PERSON, max(1, len(group) // 2))
    enroll_parts.append(group.iloc[:n_enroll])
    eval_parts.append(group.iloc[n_enroll:])

enroll_df = pd.concat(enroll_parts, ignore_index=True)
eval_df = pd.concat(eval_parts, ignore_index=True)

print("Enrollment counts:")
print(enroll_df.groupby("person").size())
print("\nEvaluation counts:")
print(eval_df.groupby("person").size())

Enrollment counts:
person
holland    24
roth       30
sopheak    30
vannet     30
vit        30
dtype: int64

Evaluation counts:
person
holland    25
roth       61
sopheak    48
vannet     42
vit        47
dtype: int64


In [10]:
# Build one prototype embedding per person (mean of enrollment embeddings)
prototypes = []
for person, group in enroll_df.groupby("person"):
    stack = np.vstack(group["embedding"].to_numpy())
    mean_emb = l2_normalize(stack.mean(axis=0))
    role_mode = group["role"].astype(str).mode()
    person_role = role_mode.iloc[0] if len(role_mode) > 0 else get_person_role(person)
    prototypes.append({
        "person": person,
        "role": person_role,
        "samples_used": len(group),
        "embedding": mean_emb,
    })

proto_df = pd.DataFrame(prototypes)
proto_matrix = np.vstack(proto_df["embedding"].to_numpy())
proto_names = proto_df["person"].tolist()
proto_roles = proto_df["role"].tolist()

print(proto_df[["person", "role", "samples_used"]])

    person     role  samples_used
0  holland  student            24
1     roth  student            30
2  sopheak  teacher            30
3   vannet  teacher            30
4      vit  student            30


In [11]:
# Identification function (cosine via NumPy, no sklearn needed)
def identify_embedding(test_embedding, threshold=COSINE_THRESHOLD):
    test_embedding = l2_normalize(test_embedding)
    sims = proto_matrix @ test_embedding
    best_idx = int(np.argmax(sims))
    best_score = float(sims[best_idx])

    if best_score >= threshold:
        return proto_names[best_idx], proto_roles[best_idx], best_score
    return "Unknown", "unknown", best_score


# Evaluate on held-out images only
if len(eval_df) > 0:
    preds = []
    for row in eval_df.itertuples(index=False):
        pred_name, pred_role, score = identify_embedding(row.embedding)
        preds.append({
            "person_true": row.person,
            "role_true": row.role,
            "person_pred": pred_name,
            "role_pred": pred_role,
            "score": score,
        })

    eval_pred_df = pd.DataFrame(preds)
    accuracy = (eval_pred_df["person_true"] == eval_pred_df["person_pred"]).mean()
    print(f"Eval accuracy (hold-out): {accuracy:.4f}")
    display(eval_pred_df.head())
else:
    print("No evaluation data after split. Increase number of images per person.")

Eval accuracy (hold-out): 0.9821


,person_true,role_true,person_pred,role_pred,score
0,holland,student,holland,student,0.796990
1,holland,student,holland,student,0.799188
2,holland,student,holland,student,0.831391
3,holland,student,holland,student,0.785632
4,holland,student,holland,student,0.846144


In [12]:
# Save CSV files
def embedding_dict(embedding, prefix="e"):
    return {f"{prefix}{i}": float(v) for i, v in enumerate(embedding)}


enroll_rows = []
for row in enroll_df.itertuples(index=False):
    item = {
        "person": row.person,
        "role": row.role,
        "image_path": row.image_path,
        "det_score": float(row.det_score),
    }
    item.update(embedding_dict(row.embedding, prefix="e"))
    enroll_rows.append(item)

enroll_csv_df = pd.DataFrame(enroll_rows)

proto_rows = []
for row in proto_df.itertuples(index=False):
    item = {
        "person": row.person,
        "role": row.role,
        "samples_used": int(row.samples_used),
    }
    item.update(embedding_dict(row.embedding, prefix="e"))
    proto_rows.append(item)

proto_csv_df = pd.DataFrame(proto_rows)

enroll_csv_path = ARTIFACTS_DIR / "enrollment_embeddings.csv"
proto_csv_path = ARTIFACTS_DIR / "person_prototypes.csv"

enroll_csv_df.to_csv(enroll_csv_path, index=False)
proto_csv_df.to_csv(proto_csv_path, index=False)

print(f"Saved: {enroll_csv_path}")
print(f"Saved: {proto_csv_path}")
print(f"Enrollment CSV shape: {enroll_csv_df.shape}")
print(f"Prototype CSV shape: {proto_csv_df.shape}")

Saved: c:\Users\chans\Documents\Sopheak\PersonalProject\fullstack\face-regcon-attendance\face_recognition_attendance_system\my-own-experiement\artifacts\enrollment_embeddings.csv
Saved: c:\Users\chans\Documents\Sopheak\PersonalProject\fullstack\face-regcon-attendance\face_recognition_attendance_system\my-own-experiement\artifacts\person_prototypes.csv
Enrollment CSV shape: (144, 516)
Prototype CSV shape: (5, 515)


In [17]:
# Experiment the datastorage first
test_image = "data/test/image2.png"
test_action = "check_in"   # "check_in" or "check_out"

In [18]:
from datetime import datetime

ATTENDANCE_UI_CSV = ARTIFACTS_DIR / "attendance_records.csv"

def _embedding_columns(df):
    emb_cols = [c for c in df.columns if c.startswith("e")]
    try:
        emb_cols = sorted(emb_cols, key=lambda x: int(x[1:]))
    except ValueError:
        emb_cols = sorted(emb_cols)
    return emb_cols

def ensure_runtime_prototypes_loaded():
    global proto_names, proto_roles, proto_matrix
    if "proto_names" in globals() and "proto_matrix" in globals() and len(proto_names) > 0:
        return

    proto_csv = ARTIFACTS_DIR / "person_prototypes.csv"
    proto_runtime_df = pd.read_csv(proto_csv)
    emb_cols = _embedding_columns(proto_runtime_df)
    proto_names = proto_runtime_df["person"].astype(str).tolist()
    if "role" in proto_runtime_df.columns:
        proto_roles = proto_runtime_df["role"].fillna("unknown").astype(str).tolist()
    else:
        proto_roles = ["unknown"] * len(proto_names)
    raw_mat = proto_runtime_df[emb_cols].to_numpy(dtype=np.float32)
    proto_matrix = np.vstack([l2_normalize(v) for v in raw_mat])

def predict_person_from_image(image_path, threshold=COSINE_THRESHOLD):
    image_path = Path(image_path)
    if not image_path.exists():
        image_path = BASE_DIR / image_path
    if not image_path.exists():
        raise FileNotFoundError(f"Test image not found: {image_path}")

    ensure_runtime_prototypes_loaded()
    out = extract_best_embedding(image_path)
    if out is None:
        return "Unknown", 0.0

    emb, _ = out
    person, _, score = identify_embedding(emb, threshold=threshold)
    return person, float(score)

def init_or_load_attendance_ui(csv_path=ATTENDANCE_UI_CSV):
    columns = [
        "date",
        "person",
        "check_in",
        "check_out",
        "status",
        "late_minutes",
        "last_similarity",
        "last_action",
        "updated_at",
    ]
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        for col in columns:
            if col not in df.columns:
                df[col] = ""
        return df[columns].copy()
    return pd.DataFrame(columns=columns)

def compute_late_minutes(check_in_time, class_start="08:00"):
    in_dt = datetime.strptime(check_in_time, "%H:%M")
    start_dt = datetime.strptime(class_start, "%H:%M")
    diff = int((in_dt - start_dt).total_seconds() // 60)
    return max(0, diff)

def upsert_attendance_record(df, person, action, similarity, class_start="08:00", now=None):
    if now is None:
        now = datetime.now()
    day = now.strftime("%Y-%m-%d")
    hhmm = now.strftime("%H:%M")

    if action not in {"check_in", "check_out"}:
        raise ValueError("action must be 'check_in' or 'check_out'")

    mask = (df["date"] == day) & (df["person"] == person)
    if mask.any():
        idx = df[mask].index[0]
    else:
        idx = len(df)
        df.loc[idx, ["date", "person", "check_in", "check_out", "status", "late_minutes"]] = [
            day, person, "", "", "Absent", 0
        ]

    if action == "check_in":
        if not str(df.loc[idx, "check_in"]):
            df.loc[idx, "check_in"] = hhmm
        effective_check_in = str(df.loc[idx, "check_in"])
        late_mins = compute_late_minutes(effective_check_in, class_start=class_start)
        df.loc[idx, "late_minutes"] = late_mins
        df.loc[idx, "status"] = "Late" if late_mins > 0 else "Present"
    else:
        df.loc[idx, "check_out"] = hhmm
        df.loc[idx, "status"] = "Left"

    df.loc[idx, "last_similarity"] = round(float(similarity), 4)
    df.loc[idx, "last_action"] = action
    df.loc[idx, "updated_at"] = now.strftime("%Y-%m-%d %H:%M:%S")
    return df

def save_attendance_ui(df, csv_path=ATTENDANCE_UI_CSV):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)
    return csv_path

# ---- Demo with one test image ----
pred_person, pred_score = predict_person_from_image(test_image)
attendance_ui_df = init_or_load_attendance_ui()

if pred_person == "Unknown":
    print("No known person recognized from test image. Attendance CSV not updated.")
else:
    attendance_ui_df = upsert_attendance_record(
        attendance_ui_df,
        person=pred_person,
        action=test_action,
        similarity=pred_score,
    )
    out_csv = save_attendance_ui(attendance_ui_df)
    print(f"Recognized: {pred_person} (similarity={pred_score:.4f})")
    print(f"Action: {test_action}")
    print(f"Saved UI attendance CSV: {out_csv}")

today = datetime.now().strftime("%Y-%m-%d")
today_view = attendance_ui_df[attendance_ui_df["date"] == today].copy()
if len(today_view) == 0:
    print("No attendance rows for today yet.")
else:
    display(today_view.sort_values(["status", "person"]).reset_index(drop=True))

c:\Users\chans\Documents\Sopheak\PersonalProject\fullstack\face-regcon-attendance\face_recognition_attendance_system\my-own-experiement\env\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


Recognized: sopheak (similarity=0.6949)
Action: check_in
Saved UI attendance CSV: c:\Users\chans\Documents\Sopheak\PersonalProject\fullstack\face-regcon-attendance\face_recognition_attendance_system\my-own-experiement\artifacts\attendance_records.csv


,date,person,check_in,check_out,status,late_minutes,last_similarity,last_action,updated_at
0,2026-03-31,sopheak,02:09,NaN,Present,0,0.6949,check_in,2026-03-31 02:19:47
